In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [4]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
    system_prompt=prompt
)

In [6]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [7]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='33c7e96e-4d61-426d-bcc8-cfe881745d29'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 156, 'prompt_tokens': 271, 'total_tokens': 427, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DHviDH6JtsVxX3WGiqDIzKJ2k6BUS', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cd8ea-d06f-7ce2-bfb9-a03167b58e3c-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'langchain-mcp-adapters'}, 'id': 'call_ERcnPRXyyrgeWLlrBp625OmT', 'type': 'tool_call'}], invalid_tool_calls=[], us

## Online MCP

In [ ]:
client = MultiServerMCPClient(
    # {
    #     "time": {
    #         "transport": "stdio",
    #         "command": "uvx",
    #         "args": [
    #             "mcp-server-time",
    #             "--local-timezone=America/New_York"
    #         ]
    #     }
    # }

    # "mcpServers": {
    #     "kiwi-com-flight-search": {
    #       "url": "https://mcp.kiwi.com"
    #     }
    #   }

    # ---uncomment for test---
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
                }
    }
)

tools = await client.get_tools()

In [14]:
agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
)

In [19]:
# question = HumanMessage(content="What time is it?")
question = HumanMessage(content="Get me a direct flight from Zagreb to Dubrovnik oneway on 15th of March for 2 adults")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='Get me a direct flight from Zagreb to Dubrovnik oneway on 15th of March for 2 adults', additional_kwargs={}, response_metadata={}, id='522b939d-1ceb-4242-91ae-2b5e0fbe3e51'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 844, 'prompt_tokens': 1222, 'total_tokens': 2066, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 768, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 1024}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DHvxkYm0Sd0gPx4VNX98bBpQH02vt', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cd8f9-80b9-7911-bc7b-decf36755eee-0', tool_calls=[{'name': 'search-flight', 'args': {'flyFrom': 'Zagreb', 'flyTo': 'Dubrovnik', 'departureDate': '15/03/2026', 'depa

In [20]:
print(response["messages"][-1].content)

Here are direct (one-way) options Zagreb (ZAG) → Dubrovnik (DBV) on 15 March for 2 adults. I’ve grouped them as requested.

Cheapest
| Route | Times | Cabin | Return route | Return times | Return cabin | Total price | Deep link |
|---|---|---|---|---|---|---|---|
| ZAG → DBV | 15/03 21:00 → 15/03 21:55 (0h 55m) | Economy | — | — | — | 155 EUR | https://on.kiwi.com/T9xaIs |

Shortest duration (all 55 minutes)
| Route | Times | Cabin | Return route | Return times | Return cabin | Total price | Deep link |
|---|---|---|---|---|---|---|---|
| ZAG → DBV | 15/03 21:00 → 15/03 21:55 (0h 55m) | Economy | — | — | — | 155 EUR | https://on.kiwi.com/T9xaIs |
| ZAG → DBV | 15/03 14:50 → 15/03 15:45 (0h 55m) | Economy | — | — | — | 187 EUR | https://on.kiwi.com/t9wYQR |
| ZAG → DBV | 15/03 11:35 → 15/03 12:30 (0h 55m) | Economy | — | — | — | 187 EUR | https://on.kiwi.com/iO0C1l |

Other direct options
| Route | Times | Cabin | Return route | Return times | Return cabin | Total price | Deep link |
|-